# Диалоговая рекомендация музыки: главный ноутбук

Вход: **диалог** (JSON, массив сообщений) + **история прослушиваний** + **инфо о пользователе** (текст).
Выход: top-k треков и текстовый ответ.

![scheme](scheme.png)

| Шаг | Модуль | Сейчас |
|---|---|---|
| 1. разбор диалога (llm-саммари) | `recsys/dialog.py` | правила (`rule`), LLM по флагу |
| 2. кандидаты | `recsys/retrieval/sources.py` | BM25 настоящий, HNSW **заглушка**, история (content-based), популярное |
| 3. RRF + фильтры + дедуп | `recsys/fusion.py` | настоящий |
| 4. ранкер | `recsys/ranking.py` | **заглушка** (порядок RRF), `heuristic`, клиент `api` |
| 5. описание от LLM | `recsys/explain.py` | **заглушка** (приветствие + список), LLM по флагу |

Код живёт в `.py`, ноутбук только импортирует и запускает. Благодаря `autoreload` правки в `.py` видны без перезапуска ядра.

Установка зависимостей (быстро, если всё уже стоит).

In [ ]:
# один раз на новой машине / после изменения requirements.txt
%pip install -q -r requirements.txt

Импорты. `autoreload` подхватывает правки в `.py` без перезапуска ядра.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

from recsys.config import load_config
from recsys.data import load_data
from recsys.pipeline import Pipeline
from recsys.schemas import Request, Message
from recsys.eval import evaluate, compare_configs

pd.set_option("display.max_colwidth", 80)

## Конфиг

Все настройки лежат в `configs/default.yaml`. Чтобы не править файл, можно переопределить нужное в `OVERRIDES` ниже.

* `data.source`: откуда брать данные. `synthetic` работает сразу, `onion` читает файлы Music4All-Onion из `data.onion.dir`, `dataset` читает наш датасет с диалогами (формат в `recsys/data/dataset.py`)
* `summarizer.type` / `explainer.type`: правила/шаблон (`rule`/`stub`) или `llm`
* `ranker.type`: `stub` | `heuristic` | `api`

In [2]:
OVERRIDES = {
    # "data": {"source": "onion", "onion": {"dir": "/home/jupyter/mnt/s3/<bucket>/onion"}},
    # "data": {"source": "dataset", "dataset": {"dir": "data/our_dataset"}},
    # "ranker": {"type": "heuristic"},
}
cfg = load_config("configs/default.yaml", OVERRIDES)
cfg["data"]["source"]

'synthetic'

## Данные

Загружаем каталог треков (теги, жанры, популярность, названия), историю прослушиваний и запросы с целевыми треками для оценки.
Таблица `stats` показывает, сколько треков с тегами и названиями.

In [3]:
data = load_data(cfg)
catalog = data.catalog
display(catalog.stats().to_frame("value").T)
catalog.df.head()

[data] synthetic: DataBundle(tracks=5000, interactions=8966, requests=300) за 0.8 c


,tracks,with_names,with_tags,with_genres,mean_tags,with_popularity,unique_tags
value,5000.0,5000.0,5000.0,5000.0,4.99,5000.0,59.0


,track_id,title,artist,tags,tag_weights,genres,popularity
0,t000000,Home Echo,Cosmic Crowns,"[rnb, romantic, sexy, smooth, soul, night, 00s]","[100.0, 78.0, 70.0, 50.0, 46.0, 16.0, 7.0]",[rnb],1.732465
1,t000001,Ocean Blue,Frozen Machines,"[metal, guitar, dark, 80s, 00s]","[100.0, 58.0, 41.0, 37.0, 34.0]",[metal],1.546442
2,t000002,Echo Rain,Frozen Machines,"[metal, energetic, aggressive, dark, male vocalists]","[100.0, 92.0, 90.0, 50.0, 34.0]",[metal],0.290494
3,t000003,Summer,Violet Owls,"[jazz, night, saxophone, smooth jazz, instrumental, 80s]","[100.0, 60.0, 49.0, 48.0, 46.0, 25.0]",[jazz],3.312242
4,t000004,Road Gold,Hollow Mirrors,"[indie rock, british, melancholic, indie, guitar, summer]","[100.0, 84.0, 81.0, 45.0, 38.0, 9.0]",[indie rock],6.329839


## Пайплайн

Собираем пять шагов по конфигу. Ниже видно, какие источники кандидатов включены и какие реализации стоят на шагах 1, 4 и 5.

In [4]:
pipe = Pipeline.from_config(cfg, catalog)
[r.name for r in pipe.retrievers], type(pipe.summarizer).__name__, type(pipe.ranker).__name__, type(pipe.explainer).__name__

(['bm25', 'hnsw', 'history', 'popular'],
 'RuleSummarizer',
 'StubRanker',
 'StubExplainer')

### Один запрос

Запрос читается из `examples/*.json`: диалог, история, инфо о пользователе. Попробуйте другие файлы из `examples/`: холодный старт, «как артист X», лайки/скипы, пустое «more».
На выходе: саммари, текст ответа и таблица треков.

In [5]:
req = Request.from_json("examples/request_01.json")
resp = pipe.run(req, debug=True)

print("SUMMARY:", resp.summary.text)
print()
print(resp.text)
resp.to_frame()

SUMMARY: User wants calm, melancholic, acoustic, folk; mood: sad; energy: low; avoid female vocalists; generally likes indie rock, folk.

Hi! Here is what I picked for you (calm, melancholic, acoustic, folk):
1. The Scarlet Foxes — Summer [folk, acoustic, calm] — matches 'calm'; matches 'melancholic'; matches 'acoustic'
2. Hollow Gardens — Road Shadows [folk, melancholic, calm] — matches 'calm'; matches 'melancholic'; matches 'acoustic'
3. The Hollow Foxes — Blue [folk, acoustic, melancholic] — matches 'calm'; matches 'melancholic'; matches 'acoustic'
4. Hollow Lights — Fire [folk, acoustic, melancholic] — matches 'calm'; matches 'melancholic'; matches 'acoustic'
5. Gentle Wolves — Smoke City [folk, acoustic, calm] — matches 'calm'; matches 'melancholic'; matches 'acoustic'
6. Scarlet Ghosts — Smoke [folk, acoustic, calm] — matches 'calm'; matches 'melancholic'; matches 'acoustic'
7. Scarlet Ghosts — Dreams [folk, acoustic, calm] — matches 'calm'; matches 'melancholic'; matches 'acoust

,rank,track_id,artist,title,score,tags,reasons
0,1,t003119,The Scarlet Foxes,Summer,0.0200,"folk, acoustic, calm, melancholic, summer",matches 'calm'; matches 'melancholic'; matches 'acoustic'
1,2,t004264,Hollow Gardens,Road Shadows,0.0164,"folk, melancholic, calm, acoustic, singer songwriter, dark",matches 'calm'; matches 'melancholic'; matches 'acoustic'
2,3,t004831,The Hollow Foxes,Blue,0.0164,"folk, acoustic, melancholic, calm",matches 'calm'; matches 'melancholic'; matches 'acoustic'
3,4,t002493,Hollow Lights,Fire,0.0161,"folk, acoustic, melancholic, calm, night",matches 'calm'; matches 'melancholic'; matches 'acoustic'
4,5,t001130,Gentle Wolves,Smoke City,0.0157,"folk, acoustic, calm, melancholic, guitar, 80s",matches 'calm'; matches 'melancholic'; matches 'acoustic'
5,6,t001291,Scarlet Ghosts,Smoke,0.0156,"folk, acoustic, calm, melancholic, singer songwriter",matches 'calm'; matches 'melancholic'; matches 'acoustic'
6,7,t002192,Scarlet Ghosts,Dreams,0.0154,"folk, acoustic, calm, melancholic, 90s",matches 'calm'; matches 'melancholic'; matches 'acoustic'
7,8,t001862,Velvet Mirrors,Dreams,0.0152,"folk, calm, guitar, melancholic, acoustic",matches 'calm'; matches 'melancholic'; matches 'acoustic'
8,9,t003697,The Cosmic Mirrors,Stars Dreams,0.0149,"folk, acoustic, calm, melancholic, guitar",matches 'calm'; matches 'melancholic'; matches 'acoustic'
9,10,t002321,Hollow Lights,Smoke,0.0148,"folk, melancholic, calm, acoustic, 90s, singer songwriter",matches 'calm'; matches 'melancholic'; matches 'acoustic'


### Что происходит на каждом шаге

`debug=True` сохраняет промежуточные результаты: саммари, сколько кандидатов дал каждый источник, веса RRF, время шагов и таблицу признаков, которую видит ранкер.

In [6]:
dbg = resp.debug
print("1. саммари:", resp.summary)
print("2. кандидатов по источникам:", dbg["n_candidates"])
print("3. веса RRF:", dbg["weights"], "| после фильтров:", dbg["n_fused"])
print("время, мс:", {k: round(v * 1000, 1) for k, v in dbg["timings"].items()})
dbg["features"].head(10)

1. саммари: DialogSummary(text='User wants calm, melancholic, acoustic, folk; mood: sad; energy: low; avoid female vocalists; generally likes indie rock, folk.', query='calm melancholic acoustic folk', include_tags=['calm', 'melancholic', 'acoustic', 'folk'], exclude_tags=['female vocalists'], seed_artists=[], exclude_artists=[], seed_track_ids=[], mood='sad', energy='low', user_tags=['indie rock', 'folk'], user_attrs={'age': 24, 'gender': 'f', 'country': 'Germany'}, source='rule')
2. кандидатов по источникам: {'bm25': 200, 'hnsw': 200, 'history': 200, 'popular': 100}
3. веса RRF: {'bm25': 1.0, 'hnsw': 0.3, 'history': 0.8, 'popular': 0.3} | после фильтров: 500
время, мс: {'1_summary': 1.2, '2_candidates': 3.6, '3_fusion': 9.2, '4_ranker': 26.0, '5_explain': 0.0}


,track_id,rrf_score,n_sources,log_popularity,query_tag_hits,query_tag_frac,user_tag_hits,profile_tag_score,artist_listened,artist_seed,n_tags,rank_bm25,score_bm25,rank_hnsw,score_hnsw,rank_history,score_history,rank_popular,score_popular,rank_score
0,t003119,0.020040,2,4.040158,4,1.0,1,0.000000,0.0,0.0,5,3,66.153244,1000,0.000000,1000,0.0,12,55.835309,0.020040
1,t004264,0.016426,2,3.158878,4,1.0,1,0.049156,0.0,0.0,6,13,63.486946,1000,0.000000,1000,0.0,50,22.544168,0.016426
2,t004831,0.016393,1,1.821842,4,1.0,1,0.000000,0.0,0.0,4,1,67.425003,1000,0.000000,1000,0.0,1000,0.000000,0.016393
3,t002493,0.016129,1,0.224095,4,1.0,1,0.000000,0.0,0.0,5,2,66.410667,1000,0.000000,1000,0.0,1000,0.000000,0.016129
4,t001130,0.015685,2,2.330232,4,1.0,1,0.113899,0.0,0.0,6,12,63.486946,107,0.009346,1000,0.0,1000,0.000000,0.015685
5,t001291,0.015625,1,1.629998,4,1.0,1,0.000000,0.0,0.0,5,4,65.258087,1000,0.000000,1000,0.0,1000,0.000000,0.015625
6,t002192,0.015385,1,0.805233,4,1.0,1,0.000000,0.0,0.0,5,5,65.051659,1000,0.000000,1000,0.0,1000,0.000000,0.015385
7,t001862,0.015152,1,1.070250,4,1.0,1,0.069538,0.0,0.0,5,6,64.802376,1000,0.000000,1000,0.0,1000,0.000000,0.015152
8,t003697,0.014925,1,0.370204,4,1.0,1,0.069538,0.0,0.0,5,7,64.577606,1000,0.000000,1000,0.0,1000,0.000000,0.014925
9,t002321,0.014804,2,0.165054,4,1.0,1,0.000000,0.0,0.0,6,15,63.220272,144,0.006944,1000,0.0,1000,0.000000,0.014804


### Несколько реплик

`next_turn` добавляет в запрос ответ ассистента, новую реплику пользователя и уже показанные треки. Показанное повторно не выдаётся, новые пожелания перекрывают старые.

In [7]:
req2 = req.next_turn(resp, "Now something more energetic, no folk")
resp2 = pipe.run(req2)
print("SUMMARY:", resp2.summary.text)
print("пересечение с прошлой выдачей:", len(set(resp.track_ids) & set(resp2.track_ids)))
resp2.to_frame()

SUMMARY: User wants melancholic, acoustic, energetic; mood: sad; energy: high; avoid female vocalists, folk; generally likes indie rock, folk.
пересечение с прошлой выдачей: 0


,rank,track_id,artist,title,score,tags,reasons
0,1,t001428,Lunar Mirrors,Blue Static,0.0131,"electronic, energetic, synth, party, dance",matches 'energetic'; you listen to Lunar Mirrors
1,2,t002913,Lunar Mirrors,City Ocean,0.0129,"electronic, energetic, house, dance, synth, summer",matches 'energetic'; you listen to Lunar Mirrors
2,3,t003927,Golden Machines,Paper Storm,0.0104,"electronic, dance, synth, happy, 80s","close to your taste: electronic, dance"
3,4,t003203,Wild Engines,Stars Home,0.0103,"electronic, synth, energetic, dance, house, sad",matches 'energetic'
4,5,t003302,Frozen Machines,Gold Static,0.0103,"metal, guitar, energetic, romantic, 80s",matches 'energetic'; you listen to Frozen Machines
5,6,t003619,Frozen Machines,Stars,0.0101,"metal, energetic, guitar",matches 'energetic'; you listen to Frozen Machines
6,7,t002674,Cosmic Owls,Silence,0.0098,"electronic, dance, energetic, synth",matches 'energetic'
7,8,t004456,Static Echoes,Ocean,0.0098,"electronic, house, dance, synth, energetic, romantic",matches 'energetic'
8,9,t000637,Hollow Crowns,Road,0.0094,"electronic, party, synth, dance, energetic, 00s",matches 'energetic'
9,10,t002416,Wild Engines,Echo,0.0087,"electronic, energetic, dance, synth, 00s",matches 'energetic'


### Запрос из датасета

Берём диалог из загруженных данных, где известно, какие треки пользователь на самом деле выбрал (`target_ids`), и смотрим, сколько из них попало в выдачу.

In [8]:
r = data.requests[0]
print([f"{m.role}: {m.text}" for m in r.dialog])
print("user_info:", r.user_info, "| история:", len(r.history), "| цели:", r.target_ids)
out = pipe.run(r)
print("попаданий в цели:", len(set(out.track_ids) & set(r.target_ids)))
out.to_frame()

['user: I want to listen to punk, preferably energetic.']
user_info: male, 37 years old, from USA. Favourite genres: folk, rnb. | история: 18 | цели: ['t000956', 't004642', 't004236', 't004427', 't002037']
попаданий в цели: 0


,rank,track_id,artist,title,score,tags,reasons
0,1,t001831,The Scarlet Foxes,Fire,0.0224,"folk, singer songwriter, acoustic",you listen to The Scarlet Foxes
1,2,t003021,The Scarlet Foxes,Static Summer,0.0180,"folk, acoustic, melancholic, singer songwriter, guitar, 00s",you listen to The Scarlet Foxes
2,3,t004583,The Midnight Wolves,Stars,0.0164,"punk, energetic, fast",matches 'punk'; matches 'energetic'
3,4,t004393,The Midnight Engines,Ocean Storm,0.0161,"punk, energetic, fast",matches 'punk'; matches 'energetic'
4,5,t003892,Silent Foxes,Silence Storm,0.0159,"punk, energetic, punk rock",matches 'punk'; matches 'energetic'
5,6,t004998,Neon Ghosts,Dreams,0.0156,"punk, energetic, fast",matches 'punk'; matches 'energetic'
6,7,t002776,The Cosmic Lights,Echo,0.0154,"punk, energetic, aggressive",matches 'punk'; matches 'energetic'
7,8,t000428,The Cosmic Lights,Static,0.0152,"punk, energetic, aggressive, sad",matches 'punk'; matches 'energetic'
8,9,t002635,Frozen Mirrors,Gold,0.0149,"punk, energetic, punk rock, aggressive",matches 'punk'; matches 'energetic'
9,10,t004091,Silent Waves,Glass,0.0145,"punk, energetic, punk rock",matches 'punk'; matches 'energetic'


## Офлайн-оценка

`evaluate` прогоняет запросы с целями и считает hit/recall/ndcg/mrr@10, а также recall каждого источника кандидатов («нашёл ли источник цель хоть где-то в своём списке»).

На синтетике и на запросах из Onion метрики завышены: запросы строятся из тегов целей. Используйте их для сравнения вариантов между собой.

In [9]:
per_request, mean = evaluate(pipe, data.requests[:200])
mean.round(3).to_frame("value").T

  50/200  0.8 c
  100/200  1.6 c
  150/200  2.4 c
  200/200  3.1 c


,hit@10,recall@10,ndcg@10,mrr@10,recall@fused,recall@bm25,recall@hnsw,recall@history,recall@popular
value,0.425,0.104,0.102,0.202,0.847,0.747,0.087,0.341,0.196


Сравнение вариантов: одни и те же запросы, разные конфиги. Видно, сколько даёт диалог, история и ранкер.

In [10]:
variants = {
    "full (stub ranker)": {},
    "full (heuristic)": {"ranker": {"type": "heuristic"}},
    "only popular": {"retrieval": {"bm25": {"enabled": False}, "hnsw": {"enabled": False}, "history": {"enabled": False}}},
    "no dialog": {"retrieval": {"bm25": {"enabled": False}, "hnsw": {"enabled": False}}},
    "no history": {"retrieval": {"history": {"enabled": False}}},
}
compare_configs(cfg, variants, catalog, data.requests[:200]).round(3)

,hit@10,mrr@10,ndcg@10,recall@10,recall@bm25,recall@fused,recall@history,recall@hnsw,recall@popular
full (stub ranker),0.425,0.202,0.102,0.104,0.747,0.847,0.341,0.087,0.196
full (heuristic),0.535,0.334,0.167,0.150,0.747,0.847,0.341,0.087,0.196
only popular,0.060,0.015,0.009,0.014,NaN,0.196,NaN,NaN,0.196
no dialog,0.250,0.098,0.059,0.070,NaN,0.428,0.341,NaN,0.196
no history,0.375,0.230,0.105,0.092,0.747,0.823,NaN,0.087,0.196


## LLM (саммари и описание)

Нужна GPU-конфигурация DataSphere и `transformers` (см. requirements.txt). Поставьте `USE_LLM = True`.
Модель загружается один раз и работает на шагах 1 и 5. Если LLM вернула не JSON, саммари берётся из правил.

In [11]:
USE_LLM = False

if USE_LLM:
    from recsys.llm import build_llm
    cfg_llm = load_config("configs/default.yaml", {
        **OVERRIDES,
        "llm": {"type": "local", "model_name": "Qwen/Qwen2.5-1.5B-Instruct"},
        "summarizer": {"type": "llm"},
        "explainer": {"type": "llm"},
    })
    llm = build_llm(cfg_llm)
    pipe_llm = Pipeline.from_config(cfg_llm, catalog, llm=llm, bm25_index=pipe.bm25_index)
    resp_llm = pipe_llm.run(req, debug=True)
    print("RAW LLM SUMMARY:", pipe_llm.summarizer.last_raw)
    print("SUMMARY:", resp_llm.summary)
    print(resp_llm.text)
    display(resp_llm.to_frame())

## Как заменить заглушку

* **HNSW**: новый класс в `recsys/retrieval/hnsw.py` с `name = "hnsw"` и методом `search(ctx) -> [Candidate]`, зарегистрировать в `build_retrievers` (`recsys/retrieval/sources.py`).
* **Ранкер**: класс с `rank(features, ctx) -> DataFrame` (колонка `rank_score`) в `recsys/ranking.py`, признаки в `build_features` там же. Для внешнего сервиса: `ranker.type: api`, `ranker.api_url`.
* **Наш датасет**: положить файлы в `data/our_dataset`, `data.source: dataset`; если поля называются иначе, поправить `data.dataset.fields`.